# Генерация датасета для классификации и оценка распределения классов

## Пролог

In [159]:
from sklearn.datasets import make_classification
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
import pandas as pd
import numpy as np

In [168]:
def get_seed():
    """
    Собственный генератор seed, который обеспечивает повторяемость, если, например, random_state нужно указывать в цикле
    :return:
    """
    return my_rng.integers(0, 2**32 - 1)


def init_seed(seed):
    global my_rng
    my_rng = np.random.default_rng(seed) # создаем собственный независимый генератор NumPy


SEED = 42       # используется в генераторе на старте. Можно и просто, если генератор не требуется
my_rng = None   # просто чтобы создать глобальную переменную
init_seed(SEED)

## Создание датасета для классификации

Параметры генерации

In [161]:
n_samples = 100  # число объектов
n_features = 5  # общее число признаков
n_informative = 5  # информативные признаки
n_redundant = 0  # линейно-зависимые признаки
n_repeated = 0  # дублирующиеся признаки
n_classes = 3  # число классов
n_clusters_per_class = 2  # кластеров на класс
class_sep = 1.5  # разделимость классов (↑ — легче)
flip_y = 0.01  # доля случайно пере-метченных меток
random_state = 42  # для воспроизводимости
weights = [0.25, 0.25, 0.50]

# print(f"Шумовых (случайных) признаков - {n_features - n_informative - n_redundant}")

In [162]:
def get_class_weights(n_samples=100, n_features=5, n_informative=3, n_classes=3, weights=[0.25, 0.25, 0.50], flip_y=0, random_state=SEED):
    X, y = make_classification(
        n_samples=n_samples,
        n_features=n_features,
        n_informative=n_informative,
        n_classes=n_classes,
        weights=weights,
        flip_y=flip_y,
        random_state=random_state,
    )
    return X, y

X, y = get_class_weights()

print("Уникальные метки и их количество:")
print(np.unique(y, return_counts=True))

df = pd.DataFrame(X, columns=[f"f{i}" for i in range(n_features)])
df["target"] = y
# print(df.head())

Уникальные метки и их количество:
(array([0, 1, 2]), array([25, 25, 50], dtype=int64))


## Разбивка и проверка на соотношение классов

В идеале при полном совпадении пропорций классов численный показатель похожести должен равняться нулю. Но он может быть около нуля потому, что вычисленные значения количества объектов в новой выборке получатся не целым и будут округлены.

### Первая попытка

In [163]:
scale = 0.7
n_splits = 2
sss = StratifiedShuffleSplit(n_splits=n_splits, train_size=scale, random_state=SEED)

for train_idx, test_idx in sss.split(X, y):
    y_small = y[train_idx]

    orig_share = np.bincount(y) / len(y)    # можно вынести из цикла
    train_share = np.bincount(y_small, minlength=len(orig_share)) / len(y_small)
    max_diff = np.abs(orig_share - train_share).max()  # максимально отличие в относительных долях классов

    threshold = 0.05  # допустимое отклонение (5 %)
    verdict = "OK" if max_diff <= threshold else "⚠️ превышено"

    print(f"Δ_max = {max_diff:.2%}  →  {verdict}")

Δ_max = 0.71%  →  OK
Δ_max = 0.71%  →  OK


### Много выборок, общая оценка

Вычислим максимальную относительнуб разницу в количестве классов основной и подвыборки для каждой пробы. Полученный массив ошибок проанализируем общими статистическими методами

Можно использовать для разбиения `get_split_y_sss` или `get_split_y_train_test`. Первая всегда будет обеспечивать соотношения классов, видно из названия `StratifiedShuffleSplit`. Вторая, традиционная `train_test_split`, обеспечит это только при использовании `stratify=y`. Соблюдается сохранение соотношения классов или нет показывает статистика массива ошибок

In [184]:
def get_split_y_sss(X, y, scale, random_state=SEED):
    sss = StratifiedShuffleSplit(n_splits=1, train_size=scale, random_state=random_state)
    y_idx = next(sss.split(X, y))[0]
    return y[y_idx]


def get_split_y_train_test(X, y, scale, random_state=None, **kwargs):

    X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=scale, random_state=random_state, **kwargs)
    return y_train


n_splits = 100
train_size = 0.7
# split_func = get_split_y_sss
split_func = get_split_y_train_test

deltas = []  # сюда кладём ошибки
init_seed(SEED)
for split in range(n_splits):
    y_small = split_func(X, y, train_size, random_state=get_seed(), stratify=y)
    orig_share = np.bincount(y) / len(y)
    train_share = np.bincount(y_small, minlength=len(orig_share)) / len(y_small)
    deltas.append(np.abs(orig_share - train_share).max())

In [185]:
print(f"Средняя погрешность = {np.mean(deltas):.2%}  +/- {np.std(deltas):.2%} ")
print(f"Максимальная погрешность = {np.max(deltas):.2%}")
print(f"Минимальная погрешность = {np.min(deltas):.2%}")
print(f"95-й перцентиль = {np.quantile(deltas, 0.95):.2%}")  # в 95 % случаев погрешность ≤ этого числа

Средняя погрешность = 0.71%  +/- 0.00% 
Максимальная погрешность = 0.71%
Минимальная погрешность = 0.71%
95-й перцентиль = 0.71%


## Выводы

1. Разбиение выборки двумя способами на практике показала как сохраняется или нет соотношение классов.
2. Получен практический опыт генерации синтетического датасета для задачи классификации
3. Научился создавать и разумно использовать собственный генератор случайных чисел для динамического создания SEED, сохраняя повторяемость